In [1]:
import os

import pandas as pd

datadir = os.path.join('data')
corr_path = os.path.join(datadir, 'correlations.csv')
corr_func = 'spearman'
dataset_path = os.path.join(datadir, 'normalized_dataset.csv')
valid_path = os.path.join(datadir, f'valid_dataset.csv')
invalid_path = os.path.join(datadir, f'invalid_dataset.csv')
basedir = "/home/ymerel/storage/results/"

In [2]:
dataset = pd.read_csv(dataset_path, delimiter=';')
valid_df = pd.read_csv(valid_path, delimiter=';')
invalid_df = pd.read_csv(invalid_path, delimiter=';')


print(f"{len(dataset)} configs in dataset")

correlations = pd.read_csv(corr_path, delimiter=';')
print(f"{len(correlations)} correlations in matrix")

matrix = correlations.pivot(index='source', columns='target', values=corr_func).fillna(1.0)

dataset.head(1010)

FileNotFoundError: [Errno 2] No such file or directory: 'data/normalized_dataset.csv'

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.tree import DecisionTreeRegressor, export_graphviz
from sklearn.model_selection import train_test_split


def predict_metric_precompiled(path, target_metric, iteration, train_sizes=[], test_size=200):
    
    results_list = []
    
    test_ds = pd.read_csv(os.path.join(path, f'sub_dataset_{test_size}_test_{iteration}.csv'), delimiter=';')
    
    ignored = [
        col for col in test_ds.columns
        if col.endswith('from_ref') or col.endswith('from_mean') or col == 'cluster' or col == 'id'
    ]
    
    y_test = test_ds[target_metric]
    X_test = test_ds.drop(columns=ignored).copy()
    
    for train_size in train_sizes:
        
        train_ds = pd.read_csv(os.path.join(path, f'sub_dataset_{train_size}_train_{iteration}.csv'), delimiter=';')
        
        y_train = train_ds[target_metric]
        X_train = train_ds.drop(columns=ignored).copy()

        regressor = DecisionTreeRegressor(random_state=None, max_depth=4)
        regressor.fit(X_train, y_train)
        y_pred = regressor.predict(X_test)
    
        mape = mean_absolute_percentage_error(y_test, y_pred)
    
        features = X_train.columns
        importances = regressor.feature_importances_
        feat_importances = {feat: imp for feat, imp in zip(features, importances) if imp > 0.0}
    
        # Store the decision tree as text
        tree_rules = export_graphviz(
                regressor,
                out_file=None,
                feature_names=X_train.columns.values,
                filled=True,
                rounded=True,
                special_characters=True,
                leaves_parallel=True,
                proportion=True
            )
    
        results_list.append({
            'metric': target_metric,
            'train_size': train_size,
            'MAPE': mape,
            'decision_tree': tree_rules,
            'feature_importances': feat_importances,
            'nb_features': len(feat_importances)
        })
    
    # Create a DataFrame from the results
    return pd.DataFrame(results_list)

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.tree import DecisionTreeRegressor, export_graphviz
from sklearn.model_selection import train_test_split


def predict_metric(dataset, target_metric, train_sizes=[], test_size=200):
    
    results_list = []
        
    train_ds, test_ds = train_test_split(dataset, test_size=test_size, random_state=None)
    ignored = [
        col for col in test_ds.columns
        if col.endswith('from_ref') or col.endswith('from_mean') or col == 'cluster' or col == 'id'
    ]
    
    y_test = test_ds[target_metric]
    X_test = test_ds.drop(columns=ignored).copy()
    
    for train_size in train_sizes:
        
        sub_train_ds, _ = train_test_split(train_ds, train_size=train_size, random_state=None)
        
        y_train = sub_train_ds[target_metric]
        X_train = sub_train_ds.drop(columns=ignored).copy()

        regressor = DecisionTreeRegressor(random_state=None, max_depth=4)
        regressor.fit(X_train, y_train)
        y_pred = regressor.predict(X_test)
    
        mape = mean_absolute_percentage_error(y_test, y_pred)
    
        features = X_train.columns
        importances = regressor.feature_importances_
        feat_importances = {feat: imp for feat, imp in zip(features, importances) if imp > 0.0}
    
        # Store the decision tree as text
        tree_rules = export_graphviz(
                regressor,
                out_file=None,
                feature_names=X_train.columns.values,
                filled=True,
                rounded=True,
                special_characters=True,
                leaves_parallel=True,
                proportion=True
            )
    
        results_list.append({
            'metric': target_metric,
            'train_size': train_size,
            'MAPE': mape,
            'decision_tree': tree_rules,
            'feature_importances': feat_importances,
            'nb_features': len(feat_importances)
        })
    
    # Create a DataFrame from the results
    return pd.DataFrame(results_list)

# Regression decision tree on full dataset
Given a configuration, try to predict the value of the correlation of this configuration result to
- the reference image
- the mean image

For prediction, means results are precomputed for each subset

In [ ]:
metrics = [f'{corr_func}_from_ref', f'{corr_func}_from_mean']

all_results = []

for i in range(1, 11):
    for metric in metrics:
        results_df = pd.DataFrame()
        if metric.endswith('from_mean'):
            results_df = predict_metric_precompiled('data/regression/full', metric, i, [80, 160, 240, 320, 400, 480, 560, 640, 720], 200)
        elif metric.endswith('from_ref'):
            results_df = predict_metric(dataset, metric, [80, 160, 240, 320, 400, 480, 560, 640, 720], 200)
        results_df['iteration'] = i
        all_results.append(results_df)

# Concatenate all results into a single DataFrame
full_results_df = pd.concat(all_results, ignore_index=True)

# Display the final DataFrame
full_results_df.head(100)

# Regression decision tree on valid dataset

In [ ]:
metrics = [f'{corr_func}_from_ref', f'{corr_func}_from_mean']

all_results = []

for i in range(1, 11):
    for metric in metrics:
        results_df = pd.DataFrame()
        if metric.endswith('from_mean'):
            results_df = predict_metric_precompiled('data/regression/valid', metric, i, [59, 118, 178, 237, 297, 356, 415, 475, 534], 149)
            results_df = pd.DataFrame()
        elif metric.endswith('from_ref'):
            results_df = predict_metric(valid_df, metric, [59, 118, 178, 237, 297, 356, 415, 475, 534], 149)
        results_df['iteration'] = i
        all_results.append(results_df)

# Concatenate all results into a single DataFrame
valid_results_df = pd.concat(all_results, ignore_index=True)

# Display the final DataFrame
valid_results_df.head(100)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- Process full_results_df ---
grouped_full = full_results_df.groupby(['metric', 'train_size'])['MAPE'].agg(['mean', 'std', 'count']).reset_index()
pivot_full = grouped_full.pivot(index='train_size', columns='metric', values=['mean', 'std', 'count'])

mean_mape_ref_full = pivot_full['mean'][f'{corr_func}_from_ref']
se_mape_ref_full = pivot_full['std'][f'{corr_func}_from_ref'] / np.sqrt(pivot_full['count'][f'{corr_func}_from_ref'])
mean_mape_mean_full = pivot_full['mean'][f'{corr_func}_from_mean']
se_mape_mean_full = pivot_full['std'][f'{corr_func}_from_mean'] / np.sqrt(pivot_full['count'][f'{corr_func}_from_mean'])

# --- Process valid_results_df ---
grouped_valid = valid_results_df.groupby(['metric', 'train_size'])['MAPE'].agg(['mean', 'std', 'count']).reset_index()
pivot_valid = grouped_valid.pivot(index='train_size', columns='metric', values=['mean', 'std', 'count'])

mean_mape_ref_valid = pivot_valid['mean'][f'{corr_func}_from_ref']
se_mape_ref_valid = pivot_valid['std'][f'{corr_func}_from_ref'] / np.sqrt(pivot_valid['count'][f'{corr_func}_from_ref'])
mean_mape_mean_valid = pivot_valid['mean'][f'{corr_func}_from_mean']
se_mape_mean_valid = pivot_valid['std'][f'{corr_func}_from_mean'] / np.sqrt(pivot_valid['count'][f'{corr_func}_from_mean'])

# --- Plotting ---
plt.figure(figsize=(10, 6))
train_sizes = np.linspace(0.2, 1.0, 9) * 0.7

# Plot for ρ from reference (full_results_df)
plt.plot(
    train_sizes,
    mean_mape_ref_full * 100,
    marker='s',  # Square marker for ref
    label=f'ρ from reference (full)',
    color='brown',  # Brown for full dataset
    linestyle='-'
)
plt.fill_between(
    train_sizes,
    (mean_mape_ref_full - se_mape_ref_full) * 100,
    (mean_mape_ref_full + se_mape_ref_full) * 100,
    alpha=0.1,
    color='brown'
)

# Plot for ρ from reference (valid_results_df)
plt.plot(
    train_sizes,
    mean_mape_ref_valid * 100,
    marker='s',  # Square marker for ref
    label=f'ρ from reference (valid)',
    color='green',  # Green for valid dataset
    linestyle='-'
)
plt.fill_between(
    train_sizes,
    (mean_mape_ref_valid - se_mape_ref_valid) * 100,
    (mean_mape_ref_valid + se_mape_ref_valid) * 100,
    alpha=0.1,
    color='green'
)

# Plot for ρ from mean (full_results_df)
plt.plot(
    train_sizes,
    mean_mape_mean_full * 100,
    marker='o',  # Round marker for mean
    label=f'ρ from mean (full)',
    color='brown',  # Brown for full dataset
    linestyle='-'
)
plt.fill_between(
    train_sizes,
    (mean_mape_mean_full - se_mape_mean_full) * 100,
    (mean_mape_mean_full + se_mape_mean_full) * 100,
    alpha=0.1,
    color='brown'
)

# Plot for ρ from mean (valid_results_df)
plt.plot(
    train_sizes,
    mean_mape_mean_valid * 100,
    marker='o',  # Round marker for mean
    label=f'ρ from mean (valid)',
    color='green',  # Green for valid dataset
    linestyle='-'
)
plt.fill_between(
    train_sizes,
    (mean_mape_mean_valid - se_mape_mean_valid) * 100,
    (mean_mape_mean_valid + se_mape_mean_valid) * 100,
    alpha=0.1,
    color='green'
)

# Annotate the last point for each line
for mean, se, color, marker, dataset, metric in zip(
    [mean_mape_ref_full, mean_mape_ref_valid, mean_mape_mean_full, mean_mape_mean_valid],
    [se_mape_ref_full, se_mape_ref_valid, se_mape_mean_full, se_mape_mean_valid],
    ['brown', 'green', 'brown', 'green'],
    ['s', 's', 'o', 'o'],
    ['full', 'valid', 'full', 'valid'],
    ['ref', 'ref', 'mean', 'mean']
):
    plt.annotate(
        f'{mean.iloc[-1] * 100:.2f} %',
        (train_sizes[-1], mean.iloc[-1] * 100),
        textcoords="offset points",
        xytext=(10, 10),
        ha='center',
        fontsize=9,
        color=color
    )
    
# Set y-axis limits to 0-50
plt.ylim(0, 45)

# Place the legend outside the figure
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlabel('Training set size')
plt.ylabel('Mean Absolute Percentage Error [MAPE] (%)')
plt.title('ρ prediction learning curves')
plt.grid(True)
plt.tight_layout()
plt.show()

# Regression decision tree on invalid dataset

In [ ]:
metrics = [f'{corr_func}_from_ref', f'{corr_func}_from_mean']

all_results = []

for i in range(1, 11):
    for metric in metrics:
        results_df = predict_metric_precompiled('data/regression/invalid', metric, i, [20, 41, 61, 82, 102, 123, 143, 164, 184], 52)
        results_df['iteration'] = i
        all_results.append(results_df)

# Concatenate all results into a single DataFrame
invalid_results_df = pd.concat(all_results, ignore_index=True)

# Display the final DataFrame
invalid_results_df.head(100)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

grouped = invalid_results_df.groupby(['metric', 'train_size'])['MAPE'].agg(['mean', 'std']).reset_index()
pivot_df = grouped.pivot(index='train_size', columns='metric', values=['mean', 'std'])

mean_mape_ref = pivot_df['mean'][f'{corr_func}_from_ref']
std_mape_ref = pivot_df['std'][f'{corr_func}_from_ref']
mean_mape_mean = pivot_df['mean'][f'{corr_func}_from_mean']
std_mape_mean = pivot_df['std'][f'{corr_func}_from_mean']

# Plotting
plt.figure(figsize=(10, 6))
train_sizes = np.linspace(0.1, 0.9, 9)

# Plot for pearson_from_ref
plt.plot(
    train_sizes,
    mean_mape_ref * 100,  # Multiply by 100 to get percentage
    marker='d',
    label=f'{corr_func}_from_ref',
    color='blue'
)
plt.fill_between(
    train_sizes,
    (mean_mape_ref - std_mape_ref) * 100,  # Multiply by 100
    (mean_mape_ref + std_mape_ref) * 100,  # Multiply by 100
    alpha=0.2,
    color='blue'
)
plt.annotate(
    f'{mean_mape_ref.iloc[-1] * 100:.2f} %',  # Multiply by 100
    (train_sizes[-1], mean_mape_ref.iloc[-1] * 100),  # Multiply by 100
    textcoords="offset points",
    xytext=(10, 10),
    ha='center',
    fontsize=10,
    color='red'
)

# Plot for pearson_from_mean
plt.plot(
    train_sizes,
    mean_mape_mean * 100,  # Multiply by 100 to get percentage
    marker='s',
    label=f'{corr_func}_from_mean',
    color='orange'
)
plt.fill_between(
    train_sizes,
    (mean_mape_mean - std_mape_mean) * 100,  # Multiply by 100
    (mean_mape_mean + std_mape_mean) * 100,  # Multiply by 100
    alpha=0.2,
    color='orange'
)
plt.annotate(
    f'{mean_mape_mean.iloc[-1] * 100:.2f} %',  # Multiply by 100
    (train_sizes[-1], mean_mape_mean.iloc[-1] * 100),  # Multiply by 100
    textcoords="offset points",
    xytext=(10, 10),
    ha='center',
    fontsize=10,
    color='red'
)

# Place the legend outside the figure
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlabel('Training Set Size')
plt.ylabel('Mean Absolute Percentage Error (MAPE) [%]')
plt.title('Learning Curve: MAPE vs Training Set Size')
plt.grid(True)
plt.tight_layout()
plt.show()